# Phase 6: SMD Multivariate Anomaly Detection

This notebook consumes experiment artifacts produced by `scripts/experiments/run_smd_multivariate.py`. It intentionally does not train models or reimplement experiment logic.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DEFAULT_RUN = (
    PROJECT_ROOT / "experiments" / "anomaly" / "smd" / "phase6_smd_multivariate_20260911_eps1e3"
)
RUN_ROOT = PROJECT_ROOT / "experiments" / "anomaly" / "smd"
RUN_DIR = DEFAULT_RUN
if not RUN_DIR.exists():
    RUN_DIR = max(RUN_ROOT.iterdir(), key=lambda path: path.stat().st_mtime)
RUN_DIR

In [ ]:
metrics = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
audit = json.loads((RUN_DIR / "dataset_audit.json").read_text(encoding="utf-8"))
global_metrics = pd.read_csv(RUN_DIR / "global_metrics.csv")
per_machine = pd.read_csv(RUN_DIR / "per_machine_metrics.csv")
runtime = pd.read_csv(RUN_DIR / "runtime.csv")
benefit = pd.read_csv(RUN_DIR / "multivariate_benefit_by_machine.csv")
global_metrics.sort_values("f1", ascending=False)

## Dataset Audit

The audit below is loaded from the run artifact. The SMD source split is preserved: source `train` is split chronologically into fit/validation, while source `test` labels are used only for final evaluation.

In [ ]:
dataset_keys = [
    "machine_count",
    "metric_count",
    "train_rows",
    "test_rows",
    "positive_test_labels",
    "positive_label_rate",
    "anomaly_segments",
    "near_constant_epsilon",
    "machines_with_train_near_constant_features",
]
{key: audit[key] for key in dataset_keys}

In [ ]:
ax = global_metrics.sort_values("f1").plot.barh(
    x="model",
    y="f1",
    figsize=(9, 4),
    legend=False,
    title="SMD global F1 by model",
)
for index, strategy in enumerate(global_metrics.sort_values("f1")["threshold_strategy"]):
    ax.text(0.01, index, strategy, va="center", color="white", fontsize=8)
ax.set_xlabel("F1")
plt.tight_layout()

In [ ]:
summary = (
    per_machine.groupby(["model", "threshold_strategy"])["f1"]
    .agg(["mean", "median", "std", lambda values: int((values == 0).sum())])
    .rename(columns={"<lambda_0>": "machines_f1_zero"})
    .sort_values("mean", ascending=False)
)
summary

In [ ]:
(
    runtime.groupby("model")[["training_seconds", "inference_seconds"]]
    .sum()
    .sort_values("training_seconds", ascending=False)
)

In [ ]:
benefit.sort_values("f1_delta_multivariate_minus_zscore", ascending=False).head(10)

In [ ]:
top_metric_path = RUN_DIR / "per_metric_signal_frequency.csv"
if top_metric_path.exists():
    top_metrics = pd.read_csv(top_metric_path)
    display(top_metrics.head(20))
else:
    print("No top-metric artifact found for this run.")

## Generated Figures

The experiment writes publication-ready PNGs under the run's `figures/` directory. Open those files directly for machine-level timelines with ground-truth regions, global scores, detections, and per-metric evidence.